# setup + imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from scipy.io import wavfile
from scipy.fft import dct

In [2]:
import librosa
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

In [3]:
from sklearn.decomposition import PCA

In [4]:
def loadwav(path):
    sr, x = wavfile.read(path)
    #print(x.dtype) -> era int16
    #print(x.ndim)
    x = x.astype(np.float32) / (2**15)
    return sr, x

In [5]:
from transformers import pipeline
asr = pipeline('automatic-speech-recognition', model='openai/whisper-base', chunk_length_s=30, device='mps')


/Users/ecaterina/practice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1121 18:37:07.880000 32381 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|█████████████████████| 245/245 [00:00<00:00, 7645.01it/s]
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


# validation

In [6]:
def evaluate(model, X_clean, y):
    scores = []
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    for fit_rows, val_rows in kf.split(X_clean, y):
        X_fit = X_clean.iloc[fit_rows]
        y_fit = y.iloc[fit_rows]
        X_val = X_clean.iloc[val_rows]
        y_val = y.iloc[val_rows]

        model.fit(X_fit, y_fit)                  
        preds = model.predict(X_val)
        scores.append(accuracy_score(y_val, preds))

    print('folds:', [round(s, 3) for s in scores])
    print('mean:', round(np.mean(scores), 4), ' spread:', round(np.std(scores), 4))
    return scores

# subtask 2

In [7]:
def extract_features(x, sr):
    if sr != 16000:
        x = librosa.resample(x, orig_sr=sr, target_sr=16000)
    mfcc = librosa.feature.mfcc(y=x, sr=16000, n_mfcc=20)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

train_data = pd.read_csv('task2/task2_train.csv', sep=';')

In [8]:
feats = []
rates = []
secs = []
for name in train_data['name']:
    sr, x = loadwav('task2/train/' + name)
    feats.append(extract_features(x, sr))
    rates.append(sr)
    secs.append(len(x) / sr)
X2 = pd.DataFrame(feats)
y2 = train_data['class']

train_data['sr'] = rates
train_data['seconds'] = secs
print(train_data.groupby('class')[['sr', 'seconds']].agg(['mean', 'min', 'max']))

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
evaluate(HistGradientBoostingClassifier(random_state=0), X2, y2)

            sr                  seconds                     
          mean    min    max       mean       min        max
class                                                       
1      24000.0  24000  24000  11.351040  2.328000  29.064000
2      16000.0  16000  16000  15.141699  3.080000  29.980000
3      22050.0  22050  22050  13.925936  2.089796  37.059048
folds: [0.989, 0.983, 0.989, 0.994, 0.978]
mean: 0.9867  spread: 0.0057


[0.9888888888888889,
 0.9833333333333333,
 0.9888888888888889,
 0.9944444444444445,
 0.9777777777777777]

In [9]:
test_names = []
for p in sorted(os.listdir('task2/test')):
    if p.endswith('.wav'):
        test_names.append(p)

feats = []
test_rates = []
for name in test_names:
    sr, x = loadwav('task2/test/' + name)
    feats.append(extract_features(x, sr))
    test_rates.append(sr)
X2_test = pd.DataFrame(feats)
print(pd.Series(test_rates).value_counts())

model = HistGradientBoostingClassifier(random_state=0).fit(X2, y2)
pred2 = model.predict(X2_test)
print(pd.Series(pred2).value_counts())

16000    200
24000    100
22050    100
Name: count, dtype: int64
1    199
2    101
3    100
Name: count, dtype: int64


# subtask 1

In [10]:
feats1 = []
secs1 = []
rates1 = []
names1 = []
for p in sorted(os.listdir('task1')):
    if p.endswith('wav'):
        sr, x = loadwav('task1/' + p)
        names1.append(p)
        rates1.append(sr)
        secs1.append(len(x)/sr)
        feats1.append(extract_features(x, sr))

X1 = pd.DataFrame(feats1)
pred1 = model.predict(X1)
for i in range(len(names1)):
    if rates1[i] == 24000: 
        pred1[i] = 1
    elif rates1[i] == 22050: 
        pred1[i] = 3
print(pd.Series(pred1).value_counts())

3    100
1    100
2    100
Name: count, dtype: int64


In [11]:
texts1 = []
for i in range(len(names1)):
    sr, x = loadwav('task1/' + names1[i])
    if sr != 16000:
        x = librosa.resample(x, orig_sr=sr, target_sr=16000)
    out = asr({'raw': x, 'sampling_rate': 16000}, generate_kwargs={'language': 'romanian', 'task': 'transcribe'})
    texts1.append(out['text'].lower().strip())
    if i < 6:
        print(names1[i], '|', texts1[i])
    if i % 50 == 0:
        print(i)
        

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


001.wav | pacienzii care urmează se efectueze oradioografie un examen prin rezonanța magnetica, computer tomograf, de obicei eu într-a se ustandardizat, vin la recepție, apoi sunt preloas de că te colegi nostri operatori.
0
002.wav | antra de verași vrea să mai amintesc aici și la apropo de inteligența artificială chiar.
003.wav | mare, dacă e acest la un glucometru de la bunis, de la cine ofii, dacă s-ați icunçat și dacă veți că perivodic, eți reți gricemiile crescute, mari. ce nu mai venea ar fi să merge la medic, sau analize din desunce.
004.wav | te niciang pot să-ți fac o prefigurare a rezultatului final și o lucrarea perfecti integrata pentru că știi cum vrei să arate știi ce structură o s-oasă aștii unde trebuie să ajun și cei mișcăr face pacientul da funcționa și cum vorbește și cum îngite și cum mă încă și cum scrășneste dinții tot tot tot și atunci dacă ai și scanare faciala ai și scanare intraorala ai și. computer tomografa și articulator digital cu toate aceste designul est

In [12]:
pd.DataFrame({'name': names1, 'text': texts1}).to_csv('texts1.csv', index=False)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = texts1

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X = vec.fit_transform(texts1)      

simil = (X @ X.T).toarray()
for i in range(len(simil)):
    simil[i, i] = -1

In [19]:
best = []
for i in range(len(simil)):
    best.append((simil[i].max(), i))
best.sort(reverse=True)

used = [0] * len(simil)
triplets = []
for b, m0 in best:
    if (used[m0]==1) :
        continue
    used[m0] = 1

    m1 = -1
    for i in range(len(simil)):
        if used[i]==0:
            if m1==-1 or simil[m0][m1] < simil[m0][i]:
                m1 = i
    used[m1] = 1

    m2 = -1
    for i in range(len(simil)):
        if used[i]==0:
            if m2==-1 or simil[m0][m2] + simil[m1][m2] < simil[m0][i] + simil[m1][i]:
                m2 = i
    used[m2] = 1
    triplets.append((m0, m1, m2))

In [20]:
rows = []
for t in range(len(triplets)):
    i, j, k = triplets[t]
    rows.append([t + 1, 1, names1[i] + '|' + names1[j] + '|' + names1[k]])

# submission

In [21]:
for i in range(len(test_names)):
    rows.append([test_names[i], 2, pred2[i]])

sub = pd.DataFrame(rows, columns=['datapointID', 'subtaskID', 'answer'])
print(sub.shape)
print(sub.head(3))
print(sub.tail(3))
sub.to_csv('submission.csv', index=False)

(500, 3)
  datapointID  subtaskID                   answer
0           1          1  256.wav|081.wav|185.wav
1           2          1  236.wav|087.wav|248.wav
2           3          1  278.wav|204.wav|068.wav
    datapointID  subtaskID answer
497     398.wav          2      3
498     399.wav          2      3
499     400.wav          2      1
